In [7]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path("/home/cristian/Documentos/Proyectos/parasol-rag-architecture")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.config import FILE_JV_FILTERED

# ─────────────────────────────────────────────────────────────
# 1. CONTEO GLOBAL
# ─────────────────────────────────────────────────────────────
df = pd.read_parquet(
    FILE_JV_FILTERED,
    columns=["cell_name", "curve", "is_curve_valid"],
)

n_points     = len(df)
n_curves     = df.groupby(["cell_name", "curve"]).ngroups
n_valid      = df[df["is_curve_valid"] == 1].groupby(["cell_name", "curve"]).ngroups
n_rejected   = n_curves - n_valid
rej_rate     = n_rejected / n_curves if n_curves else 0
pts_per_curve = n_points / n_curves if n_curves else 0

print("=" * 60)
print(" CONTEO GLOBAL")
print("=" * 60)
print(f" Puntos totales (filas)     : {n_points:>12,}")
print(f" Curvas únicas              : {n_curves:>12,}")
print(f" Curvas válidas             : {n_valid:>12,}")
print(f" Curvas rechazadas          : {n_rejected:>12,}  ({rej_rate:.2%})")
print(f" Puntos por curva (media)   : {pts_per_curve:>12.1f}")

# ─────────────────────────────────────────────────────────────
# 2. DESGLOSE POR CÉLULA
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(" POR CÉLULA")
print("=" * 60)

per_cell = (
    df.groupby("cell_name")
      .apply(lambda g: pd.Series({
          "curvas_totales": g.groupby("curve").ngroups,
          "curvas_validas": g[g["is_curve_valid"] == 1].groupby("curve").ngroups,
      }))
)
per_cell["rechazadas"] = per_cell["curvas_totales"] - per_cell["curvas_validas"]
per_cell["tasa_rechazo"] = per_cell["rechazadas"] / per_cell["curvas_totales"]
print(per_cell.to_string())

# ─────────────────────────────────────────────────────────────
# 3. DESGLOSE POR REGLA (necesita todas las columnas de QC)
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(" DESGLOSE POR REGLA DE FILTRADO")
print("=" * 60)

# Recargamos con las columnas de reglas
rules = [
    "is_low_voltage", "is_low_current", "is_night_time",
    "is_curve_frozen", "is_corrupted", "is_spike_error",
    "is_asymmetric", "is_mean_mismatch",
]

df_full = pd.read_parquet(
    FILE_JV_FILTERED,
    columns=["cell_name", "curve", "is_curve_valid"] + rules,
)

# Una fila por curva: max() consolida los flags (constantes dentro de una curva)
curve_level = df_full.groupby(["cell_name", "curve"], sort=False).max(numeric_only=True)
total = len(curve_level)

print(f" Base: {total:,} curvas\n")
print(f" {'Regla':<22} {'# Rechaza':>12} {'% del total':>12}")
print("-" * 60)
for r in rules:
    n = int(curve_level[r].sum())
    print(f" {r:<22} {n:>12,} {n/total:>11.2%}")

# ─────────────────────────────────────────────────────────────
# 4. SOLAPAMIENTO ENTRE REGLAS
# ─────────────────────────────────────────────────────────────
rejected = curve_level[curve_level["is_curve_valid"] == 0]
n_rules_per_curve = rejected[rules].sum(axis=1)

print("\n" + "=" * 60)
print(" SOLAPAMIENTO (cuántas reglas disparan por curva rechazada)")
print("=" * 60)
for k in range(1, len(rules) + 1):
    n = int((n_rules_per_curve == k).sum())
    if n:
        print(f" {k} regla(s) disparada(s): {n:>10,}  ({n/len(rejected):.2%} de las rechazadas)")

# ─────────────────────────────────────────────────────────────
# 5. REGLAS QUE "SIEMPRE VAN JUNTAS" (redundancia)
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(" MATRIZ DE CO-OCURRENCIA (sobre curvas rechazadas)")
print("=" * 60)
cooc = rejected[rules].T.astype(int).dot(rejected[rules].astype(int))
print(cooc.to_string())
print("\nLectura: diagonal = veces que la regla aparece;")
print("off-diagonal[i,j] = veces que i y j aparecen juntas en la misma curva.")

 CONTEO GLOBAL
 Puntos totales (filas)     :   28,336,095
 Curvas únicas              :      104,367
 Curvas válidas             :       49,688
 Curvas rechazadas          :       54,679  (52.39%)
 Puntos por curva (media)   :        271.5

 POR CÉLULA
            curvas_totales  curvas_validas  rechazadas  tasa_rechazo
cell_name                                                           
A162ALXP01           10218            7082        3136      0.306909
A164AB302c           10382            7112        3270      0.314968
A167AB344a           10380            7219        3161      0.304528
A170AB302             2277            1376         901      0.395696
ASLRXAB341           10192            7352        2840      0.278650
M0                   22236            4464       17772      0.799244
M83AB302              4427            1556        2871      0.648520
P12                  34255           13527       20728      0.605109

 DESGLOSE POR REGLA DE FILTRADO
 Base: 104,367 curvas

 

In [8]:
"""
Diagnóstico: ¿el filtro rechaza células degradadas (falsos positivos) o fallos de medida?
"""
import numpy as np
import pandas as pd
from src.config import FILE_JV_FILTERED, CELL_AREA_M2

CELL_AREA_CM2 = CELL_AREA_M2 * 1e4

def raw_voc_jsc_ff(V, I):
    """Extrae Voc, Jsc, pseudo-FF de una curva cruda (sin usar el filtro)."""
    idx = np.argsort(V)
    V, I = V[idx], I[idx]
    
    # dedupe V
    V_u, inv = np.unique(V, return_inverse=True)
    if len(V_u) < 5:
        return np.nan, np.nan, np.nan
    I_u = np.bincount(inv, weights=I) / np.bincount(inv)
    
    # Jsc = I at V=0
    if V_u.min() <= 0 <= V_u.max():
        jsc_A = np.interp(0.0, V_u, I_u)
    else:
        jsc_A = I_u[np.argmin(np.abs(V_u))]
    jsc = jsc_A * 1000.0 / CELL_AREA_CM2  # mA/cm^2
    
    # Voc = V where I crosses 0, V > 0.1
    sign_I = np.sign(I_u)
    changes = np.where(np.diff(sign_I) != 0)[0]
    changes = changes[V_u[changes] > 0.1]
    if len(changes) > 0:
        i = changes[0]
        v0, v1 = V_u[i], V_u[i+1]
        i0, i1 = I_u[i], I_u[i+1]
        voc = float(v0 - i0 * (v1 - v0) / (i1 - i0)) if i1 != i0 else float(v0)
    else:
        voc = float(V_u[np.argmin(np.abs(I_u))])
    
    # Pseudo-FF: max(V*I) / (Voc * Jsc_A)
    # Usar A (no mA/cm²) para consistencia
    P = V_u * I_u
    if voc > 0 and abs(jsc_A) > 0:
        ff = float(P.max() / (voc * abs(jsc_A)))
    else:
        ff = np.nan
    
    return voc, jsc, ff


def sample_and_evaluate(df, n_per_group=300, seed=42):
    """Samplea n curvas por (cell_name, is_curve_valid) y extrae raw params."""
    rng = np.random.RandomState(seed)
    rows = []
    
    for (cell, valid), group in df.groupby(["cell_name", "is_curve_valid"], sort=False):
        curve_ids = group["curve"].unique()
        if len(curve_ids) > n_per_group:
            curve_ids = rng.choice(curve_ids, n_per_group, replace=False)
        
        sub = group[group["curve"].isin(curve_ids)]
        
        # Group by curve and extract
        for cid, c in sub.groupby("curve", sort=False):
            v = c["Voltage_V"].to_numpy()
            i = c["Current_A"].to_numpy()
            voc, jsc, ff = raw_voc_jsc_ff(v, i)
            rows.append({
                "cell_name": cell,
                "is_curve_valid": int(valid),
                "curve": cid,
                "n_points": len(c),
                "voc_raw": voc,
                "jsc_raw": jsc,
                "ff_raw": ff,
                "v_span_raw": float(np.nanmax(v) - np.nanmin(v)),
                "i_span_raw": float(np.nanmax(i) - np.nanmin(i)),
            })
    
    return pd.DataFrame(rows)


def main():
    print("Cargando FILE_JV_FILTERED (solo columnas necesarias)...")
    df = pd.read_parquet(
        FILE_JV_FILTERED,
        columns=["cell_name", "curve", "is_curve_valid", "Voltage_V", "Current_A"]
    )
    print(f"Filas totales: {len(df):,}")
    print(f"Células: {df['cell_name'].unique().tolist()}")
    print(f"Grupo α (rechazo razonable): A162ALXP01, A164AB302c, A167AB344a, A170AB302, ASLRXAB341")
    print(f"Grupo β (rechazo anómalo): M0, M83AB302, P12")
    
    print("\nSampleando curvas y extrayendo parámetros crudos...")
    params = sample_and_evaluate(df, n_per_group=300)
    print(f"Total curvas analizadas: {len(params)}")
    
    # Guardar para inspección
    params.to_csv("diagnostic_raw_params.csv", index=False)
    print("Guardado: diagnostic_raw_params.csv\n")
    
    # ---- TABLA 1: métricas crudas por célula × válida ----
    print("=" * 100)
    print(" TABLA 1 — VOC, JSC, FF CRUDOS POR (célula, válida)")
    print("=" * 100)
    summary = params.groupby(["cell_name", "is_curve_valid"]).agg(
        n_curvas=("curve", "count"),
        voc_med=("voc_raw", "median"),
        voc_p10=("voc_raw", lambda x: x.quantile(0.10)),
        voc_p90=("voc_raw", lambda x: x.quantile(0.90)),
        jsc_med=("jsc_raw", "median"),
        jsc_p10=("jsc_raw", lambda x: x.quantile(0.10)),
        jsc_p90=("jsc_raw", lambda x: x.quantile(0.90)),
        ff_med=("ff_raw", "median"),
        v_span_med=("v_span_raw", "median"),
        i_span_med=("i_span_raw", "median"),
    ).round(3)
    print(summary.to_string())
    
    # ---- TABLA 2: foco en el delta aceptada vs rechazada ----
    print("\n" + "=" * 100)
    print(" TABLA 2 — DELTA (rechazada − aceptada) POR CÉLULA")
    print("=" * 100)
    print(" Lectura: si Δvoc ≈ 0 y Δjsc ≈ 0, el filtro rechaza células que ")
    print(" generan corriente/voc normal → FALSOS POSITIVOS.")
    print(" Si Δvoc << 0 y Δjsc << 0, el filtro rechaza basura → OK.\n")
    
    pivot = params.groupby(["cell_name", "is_curve_valid"]).agg(
        voc=("voc_raw", "median"),
        jsc=("jsc_raw", "median"),
        ff=("ff_raw", "median"),
    ).unstack("is_curve_valid")
    
    delta = pd.DataFrame({
        "voc_aceptada": pivot[("voc", 1)],
        "voc_rechazada": pivot[("voc", 0)],
        "Δvoc": pivot[("voc", 0)] - pivot[("voc", 1)],
        "jsc_aceptada": pivot[("jsc", 1)],
        "jsc_rechazada": pivot[("jsc", 0)],
        "Δjsc": pivot[("jsc", 0)] - pivot[("jsc", 1)],
        "ff_aceptada": pivot[("ff", 1)],
        "ff_rechazada": pivot[("ff", 0)],
        "Δff": pivot[("ff", 0)] - pivot[("ff", 1)],
    }).round(3)
    print(delta.to_string())
    
    # ---- TABLA 3: ratio de rechazo según reglas, agrupado por grupo α/β ----
    print("\n" + "=" * 100)
    print(" TABLA 3 — ¿Las rechazadas tienen Voc/Jsc razonables?")
    print("=" * 100)
    
    # Definir "razonable" como Voc > 0.5 V y Jsc > 5 mA/cm² (umbrales físicos)
    rejected = params[params["is_curve_valid"] == 0].copy()
    rejected["voc_ok"] = rejected["voc_raw"] > 0.5
    rejected["jsc_ok"] = rejected["jsc_raw"] > 5.0
    rejected["both_ok"] = rejected["voc_ok"] & rejected["jsc_ok"]
    
    razon = rejected.groupby("cell_name").agg(
        rechazadas=("curve", "count"),
        voc_razonable_pct=("voc_ok", "mean"),
        jsc_razonable_pct=("jsc_ok", "mean"),
        ambos_ok_pct=("both_ok", "mean"),
    ).round(3)
    razon["voc_razonable_pct"] *= 100
    razon["jsc_razonable_pct"] *= 100
    razon["ambos_ok_pct"] *= 100
    print(razon.to_string())
    
    print("\n" + "=" * 100)
    print(" INTERPRETACIÓN")
    print("=" * 100)
    print("""
 Si en alguna célula el % de 'ambos_ok_pct' es alto (>30%), esas rechazadas
 NO son basura: son células generando Voc/Jsc normales que el filtro está
 descartando. Eso es FALSO POSITIVO masivo.
 
 Si M0/M83/P12 tienen 'ambos_ok_pct' bajo (<10%), el filtro funciona: 
 esas células tienen hardware real problem y el rechazo es correcto.
 
 Comparativa clave: 
 - Grupo α debería tener ambos_ok_pct BAJO (pocas rechazadas son válidas)
 - Si grupo β tiene ambos_ok_pct ALTO, el filtro está castigando células
   degradadas o canales ADC distintos, no fallos de medida.
""")


if __name__ == "__main__":
    main()

Cargando FILE_JV_FILTERED (solo columnas necesarias)...
Filas totales: 28,336,095
Células: ['A162ALXP01', 'A164AB302c', 'A167AB344a', 'A170AB302', 'ASLRXAB341', 'M0', 'M83AB302', 'P12']
Grupo α (rechazo razonable): A162ALXP01, A164AB302c, A167AB344a, A170AB302, ASLRXAB341
Grupo β (rechazo anómalo): M0, M83AB302, P12

Sampleando curvas y extrayendo parámetros crudos...
Total curvas analizadas: 4800
Guardado: diagnostic_raw_params.csv

 TABLA 1 — VOC, JSC, FF CRUDOS POR (célula, válida)
                           n_curvas  voc_med  voc_p10  voc_p90  jsc_med  jsc_p10  jsc_p90  ff_med  v_span_med  i_span_med
cell_name  is_curve_valid                                                                                                
A162ALXP01 0                    300    0.087    0.031    0.315    0.498    0.000    1.492   0.454       0.004       0.000
           1                    300    0.587    0.415    0.678    8.334    3.467   17.801   0.401       0.588       0.006
A164AB302c 0          

In [9]:
import pandas as pd
from src.config import FILE_JV_FILTERED, DEPLOYMENT_TIMEZONE

df = pd.read_parquet(
    FILE_JV_FILTERED,
    columns=["cell_name", "curve", "is_curve_valid", "Timestamp"],
)
df["hour_local"] = df["Timestamp"].dt.tz_convert(DEPLOYMENT_TIMEZONE).dt.hour

# Una hora por curva (cualquiera sirve, son todas iguales dentro de la curva)
first = df.groupby(["cell_name", "curve"]).agg(
    valid=("is_curve_valid", "first"),
    hour=("hour_local", "first"),
).reset_index()

# Distribución horaria de RECHAZADAS por célula
print("\n% de rechazos por hora local — si se concentra en 6-8h y 20-22h, es baja luz:")
tabla = (
    first.groupby(["cell_name", "hour"])["valid"]
    .agg(rechazo_pct=lambda x: 100 * (1 - x.mean()))
    .unstack("hour")
    .round(0)
)
print(tabla.to_string())


% de rechazos por hora local — si se concentra en 6-8h y 20-22h, es baja luz:
           rechazo_pct                                                                                                          
hour                0      7      8     9     10    11    12    13    14    15    16    17    18    19    20    21     22     23
cell_name                                                                                                                       
A162ALXP01       100.0  100.0   95.0  81.0  31.0   8.0   2.0   0.0   0.0   0.0   1.0   3.0   4.0   9.0  23.0  34.0   92.0  100.0
A164AB302c       100.0  100.0   98.0  80.0  32.0   8.0   2.0   0.0   0.0   0.0   2.0   2.0   6.0  15.0  27.0  44.0   82.0  100.0
A167AB344a       100.0   99.0   93.0  74.0  32.0   9.0   4.0   3.0   2.0   2.0   3.0   4.0   6.0  14.0  24.0  29.0   79.0  100.0
A170AB302          NaN  100.0   82.0  59.0  37.0  28.0  25.0  17.0   7.0   7.0   3.0   3.0  21.0  61.0  90.0  83.0   98.0  100.0
ASLRXAB341       1

In [11]:
"""
audit_fn_A_diode.py — FN rate por ajuste de diodo.
Solo lee FILE_JV_FILTERED. No toca el filtro.
"""
import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from src.config import FILE_JV_FILTERED, CELL_AREA_M2

CELL_AREA_CM2 = CELL_AREA_M2 * 1e4
N_SAMPLE = 500           # curvas aceptadas a muestrear por célula
RESIDUAL_TOL = 0.05      # 5% de error normalizado por Jsc

def fit_single_diode(V, I_mA):
    """Devuelve (params, residual_norm, converged)."""
    V_t = 0.02585
    J_scale = max(abs(I_mA[0]), 1e-3)   # normalizador: Jsc aprox
    
    def residuals(p):
        J_ph, J_0, n, R_s, R_sh = p
        V_j = V + I_mA * R_s / 1000.0   # caída en R_s (mA/cm² * Ω·cm² / 1000)
        J_pred = J_ph - J_0 * (np.exp(V_j / (n * V_t)) - 1) - V_j / R_sh
        return (J_pred - I_mA) / J_scale
    
    p0 = [I_mA[0], 1e-9, 1.3, 0.01, 500.0]
    bounds = ([0, 0, 1.0, 0, 1.0], [np.inf, 1e-6, 2.5, 10.0, 1e6])
    
    try:
        r = least_squares(residuals, p0, bounds=bounds, method='trf', max_nfev=200)
        rmse = float(np.sqrt(np.mean(r.fun**2)))
        return r.x, rmse, r.success
    except Exception:
        return None, np.inf, False

def audit_curve(V, I_A):
    V = np.asarray(V, dtype=float)
    I_mA = np.asarray(I_A, dtype=float) * 1000.0 / CELL_AREA_CM2
    # Ordenar y promediar duplicados
    idx = np.argsort(V); V, I_mA = V[idx], I_mA[idx]
    V_u, inv = np.unique(V, return_inverse=True)
    if len(V_u) < 10:
        return {"fn_diode": True, "reason": "few_points"}
    I_u = np.bincount(inv, weights=I_mA) / np.bincount(inv)
    
    _, rmse, ok = fit_single_diode(V_u, I_u)
    fn = (not ok) or (rmse > RESIDUAL_TOL)
    return {"fn_diode": fn, "residual": rmse, "converged": ok}

# --- Main ---
df = pd.read_parquet(
    FILE_JV_FILTERED,
    columns=["cell_name", "curve", "is_curve_valid", "Voltage_V", "Current_A"],
)
accepted = df[df["is_curve_valid"] == 1]

results = []
for cell, g in accepted.groupby("cell_name", sort=False):
    curves = g["curve"].unique()
    n = min(len(curves), N_SAMPLE)
    chosen = np.random.RandomState(42).choice(curves, n, replace=False)
    for cid in chosen:
        c = g[g["curve"] == cid]
        r = audit_curve(c["Voltage_V"].values, c["Current_A"].values)
        r.update({"cell_name": cell, "curve": cid})
        results.append(r)

res = pd.DataFrame(results)
res.to_csv("audit_fn_diode.csv", index=False)

print("\n=== TEST A — FN rate por diodo ===")
print(res.groupby("cell_name")["fn_diode"].agg(["mean", "sum", "count"]).round(3))
print(f"\nGlobal FN rate: {res['fn_diode'].mean():.2%}")
print(f"Curvas sospechosas: {res['fn_diode'].sum()} de {len(res)}")


=== TEST A — FN rate por diodo ===
             mean  sum  count
cell_name                    
A162ALXP01  1.000  500    500
A164AB302c  1.000  500    500
A167AB344a  0.996  498    500
A170AB302   1.000  500    500
ASLRXAB341  1.000  500    500
M0          1.000  500    500
M83AB302    0.932  466    500
P12         1.000  500    500

Global FN rate: 99.10%
Curvas sospechosas: 3964 de 4000


In [12]:
"""
audit_fn_B_irradiance.py — FN rate por coherencia PCE vs POA.
Necesita joinear con FILE_TELEMETRY_10MIN.
"""
import numpy as np
import pandas as pd
from src.config import FILE_JV_FILTERED, FILE_TELEMETRY_10MIN, CELL_AREA_M2

# 1. Cargar curvas aceptadas + Timestamp
jv = pd.read_parquet(
    FILE_JV_FILTERED,
    columns=["cell_name", "curve", "is_curve_valid", "Timestamp",
             "Voltage_V", "Current_A"],
)
accepted = jv[jv["is_curve_valid"] == 1].copy()

# 2. Pmax por curva (potencia en MPP aprox = max(V*I))
pmax = accepted.groupby(["cell_name", "curve"]).apply(
    lambda g: float((g["Voltage_V"] * g["Current_A"]).max()),
    include_groups=False,
).rename("P_max_W").reset_index()

# 3. Timestamp medio por curva (para joinear con meteo)
tcurve = accepted.groupby(["cell_name", "curve"])["Timestamp"].mean().rename("ts").reset_index()
pmax = pmax.merge(tcurve, on=["cell_name", "curve"])

# 4. Cargar meteo 10-min y joinear por asof
meteo = pd.read_parquet(FILE_TELEMETRY_10MIN,
                        columns=["Timestamp", "POA_Irradiance_W_m2"])
meteo = meteo.sort_values("Timestamp").set_index("Timestamp")

pmax = pmax.sort_values("ts")
merged = pd.merge_asof(
    pmax, meteo.reset_index().sort_values("Timestamp"),
    left_on="ts", right_on="Timestamp", direction="nearest",
    tolerance=pd.Timedelta("20min"),
)

# 5. PCE esperada vs POA
mask = merged["POA_Irradiance_W_m2"] > 200   # solo mediodía
merged.loc[mask, "PCE_pct"] = (
    merged.loc[mask, "P_max_W"]
    / (merged.loc[mask, "POA_Irradiance_W_m2"] * CELL_AREA_M2)
    * 100.0
)

# 6. FN: PCE > 30% (imposible para perovskita) o PCE < 0.5% con POA alto
merged["fn_irr"] = (
    (merged["PCE_pct"] > 30.0)          # PCE imposible alto → probable artefacto
    | ((merged["PCE_pct"] < 0.5) & mask) # PCE ridículo con sol → probable basura
)

merged.to_csv("audit_fn_irradiance.csv", index=False)

print("\n=== TEST B — FN rate por coherencia con POA ===")
print(merged[mask].groupby("cell_name")["fn_irr"].agg(["mean", "sum", "count"]).round(3))
print(f"\nGlobal FN rate (solo POA>200): {merged.loc[mask, 'fn_irr'].mean():.2%}")
print(f"Curvas con PCE > 30%: {(merged['PCE_pct'] > 30).sum()}")
print(f"Curvas con PCE < 0.5% (POA>200): {((merged['PCE_pct'] < 0.5) & mask).sum()}")


=== TEST B — FN rate por coherencia con POA ===
             mean   sum  count
cell_name                     
A162ALXP01  0.001     5   6081
A164AB302c  0.000     1   6136
A167AB344a  0.000     1   6017
A170AB302   0.003     4   1246
ASLRXAB341  0.000     1   6094
M0          0.342  1286   3762
M83AB302    0.040    52   1312
P12         0.019   195  10239

Global FN rate (solo POA>200): 3.78%
Curvas con PCE > 30%: 4
Curvas con PCE < 0.5% (POA>200): 1541


In [13]:
"""
audit_fn_C_neighbors.py — Una curva aceptada debe parecerse a sus vecinas cercanas.
FN: curvas que son outliers dentro de su propia célula y hora.
"""
import numpy as np
import pandas as pd
from src.config import FILE_JV_FILTERED

jv = pd.read_parquet(
    FILE_JV_FILTERED,
    columns=["cell_name", "curve", "is_curve_valid", "Timestamp",
             "Voltage_V", "Current_A"],
)
accepted = jv[jv["is_curve_valid"] == 1].copy()

# Features por curva
def curve_feats(g):
    V = g["Voltage_V"].to_numpy()
    I = g["Current_A"].to_numpy()
    return pd.Series({
        "jsc": float(np.interp(0, np.sort(V), I[np.argsort(V)])),
        "pmax": float((V * I).max()),
        "vspan": float(V.max() - V.min()),
        "ispan": float(I.max() - I.min()),
    })

feats = accepted.groupby(["cell_name", "curve"]).apply(curve_feats, include_groups=False).reset_index()
ts = accepted.groupby(["cell_name", "curve"])["Timestamp"].mean().rename("ts").reset_index()
feats = feats.merge(ts, on=["cell_name", "curve"])

# Hora local
feats["hour"] = pd.to_datetime(feats["ts"], utc=True).dt.tz_convert("Europe/Madrid").dt.hour

# Para cada (célula, hora), ¿cuánto se desvía cada curva de la mediana?
def mad_z(s):
    med = s.median()
    mad = (s - med).abs().median() + 1e-12
    return (s - med).abs() / (1.4826 * mad)   # z-score robusto

feats["z_jsc"] = feats.groupby(["cell_name", "hour"])["jsc"].transform(mad_z)
feats["z_pmax"] = feats.groupby(["cell_name", "hour"])["pmax"].transform(mad_z)

# FN: alguna métrica con z > 4 (outlier robusto dentro de su contexto)
feats["fn_neighbor"] = (feats["z_jsc"] > 4) | (feats["z_pmax"] > 4)

feats.to_csv("audit_fn_neighbors.csv", index=False)

print("\n=== TEST C — FN rate por outlier temporal ===")
print(feats.groupby("cell_name")["fn_neighbor"].agg(["mean", "sum", "count"]).round(3))
print(f"\nGlobal FN rate: {feats['fn_neighbor'].mean():.2%}")
print(f"Curvas outlier: {feats['fn_neighbor'].sum()} de {len(feats)}")


=== TEST C — FN rate por outlier temporal ===
             mean  sum  count
cell_name                    
A162ALXP01  0.029  203   7082
A164AB302c  0.067  480   7112
A167AB344a  0.057  413   7219
A170AB302   0.000    0   1376
ASLRXAB341  0.061  452   7352
M0          0.130  580   4464
M83AB302    0.129  200   1556
P12         0.050  672  13527

Global FN rate: 6.04%
Curvas outlier: 3000 de 49688


In [14]:
"""
audit_fn_D_outliers.py — Isolation Forest sobre features físicas aceptadas.
NO para detectar degradación — solo para detectar colas extremas.
"""
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from src.config import FILE_JV_FILTERED

jv = pd.read_parquet(
    FILE_JV_FILTERED,
    columns=["cell_name", "curve", "is_curve_valid",
             "Voltage_V", "Current_A"],
)
accepted = jv[jv["is_curve_valid"] == 1]

def curve_feats(g):
    V = g["Voltage_V"].to_numpy()
    I = g["Current_A"].to_numpy()
    Vs, Is = V[np.argsort(V)], I[np.argsort(V)]
    return pd.Series({
        "vspan": V.max() - V.min(),
        "ispan": I.max() - I.min(),
        "jsc": float(np.interp(0, Vs, Is)),
        "pmax": float((V * I).max()),
        "monotonicity": float(np.mean(np.diff(Is) < 0)),  # fracción decreciente
        "n_points": len(V),
    })

# Muestrear 20k curvas aceptadas para no explotar
sample_curves = accepted.groupby("cell_name")["curve"].apply(
    lambda x: x.sample(min(len(x), 2500), random_state=42)
).values
sample_curves = set(sample_curves)
mask = accepted["curve"].isin(sample_curves)
feats = accepted[mask].groupby(["cell_name", "curve"]).apply(
    curve_feats, include_groups=False
).reset_index()

feature_cols = ["vspan", "ispan", "jsc", "pmax", "monotonicity", "n_points"]
X = feats[feature_cols].fillna(0).values

iso = IsolationForest(contamination=0.02, random_state=42, n_estimators=200)
feats["if_outlier"] = iso.fit_predict(X) == -1
feats["if_score"] = iso.score_samples(X)

feats.to_csv("audit_fn_outliers.csv", index=False)

print("\n=== TEST D — FN rate por outlier multivariante ===")
print(feats.groupby("cell_name")["if_outlier"].agg(["mean", "sum", "count"]).round(3))
print(f"\nGlobal FN rate: {feats['if_outlier'].mean():.2%}")


=== TEST D — FN rate por outlier multivariante ===
             mean  sum  count
cell_name                    
A162ALXP01  0.010   54   5486
A164AB302c  0.008   43   5587
A167AB344a  0.003   19   5587
A170AB302   0.017   23   1365
ASLRXAB341  0.003   15   5733
M0          0.103  371   3589
M83AB302    0.065   99   1513
P12         0.013   81   6371

Global FN rate: 2.00%


In [15]:
import pandas as pd

a = pd.read_csv("audit_fn_diode.csv")[["cell_name", "curve", "fn_diode"]]
b = pd.read_csv("audit_fn_irradiance.csv")[["cell_name", "curve", "fn_irr"]]
c = pd.read_csv("audit_fn_neighbors.csv")[["cell_name", "curve", "fn_neighbor"]]
d = pd.read_csv("audit_fn_outliers.csv")[["cell_name", "curve", "if_outlier"]]

m = a.merge(b, on=["cell_name", "curve"], how="outer") \
     .merge(c, on=["cell_name", "curve"], how="outer") \
     .merge(d, on=["cell_name", "curve"], how="outer").fillna(False)

m["n_votes"] = m[["fn_diode", "fn_irr", "fn_neighbor", "if_outlier"]].sum(axis=1)

print("\n=== CONSENSO DE JUECES ===")
print(m["n_votes"].value_counts().sort_index())
print(f"\n4 votos (FN casi seguro):     {(m['n_votes']==4).sum()}")
print(f"3 votos (FN probable):        {(m['n_votes']==3).sum()}")
print(f"2 votos (FN posible):         {(m['n_votes']==2).sum()}")
print(f"0-1 votos (sanas):            {(m['n_votes']<=1).sum()}")

m.to_csv("audit_fn_consensus.csv", index=False)


=== CONSENSO DE JUECES ===
n_votes
0    41265
1     7689
2      678
3       55
4        1
Name: count, dtype: int64

4 votos (FN casi seguro):     1
3 votos (FN probable):        55
2 votos (FN posible):         678
0-1 votos (sanas):            48954
